# Space Invaders — entrenamiento en Colab (GPU)

Entrena el agente Rainbow-lite de este repositorio sobre una GPU de Colab.

**Antes de ejecutar:** `Entorno de ejecución` → `Cambiar tipo de entorno de ejecución` → `GPU`.
Sin eso el notebook corre en CPU y es ~20 veces más lento.

La corrida guarda checkpoints en tu Google Drive cada 100 000 pasos. Si Colab se
desconecta (pasa seguido en la versión gratuita), basta con **volver a ejecutar
todo**: la celda de entrenamiento usa `--reanudar --hasta`, así que retoma donde
se quedó en vez de empezar de cero.

## 1. Verificar que hay GPU

In [ ]:
!nvidia-smi
import torch
print("torch", torch.__version__, "| CUDA disponible:", torch.cuda.is_available())
assert torch.cuda.is_available(), (
    "No hay GPU. Entorno de ejecución -> Cambiar tipo de entorno de ejecución -> GPU"
)
print("GPU:", torch.cuda.get_device_name(0))

## 2. Montar Google Drive

Los checkpoints y los CSV de métricas viven en Drive, no en el disco de Colab:
el disco de Colab se borra al desconectarse y se perderían horas de entrenamiento.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
DIR_DRIVE = Path('/content/drive/MyDrive/space-invaders-dpl')
(DIR_DRIVE / 'modelos').mkdir(parents=True, exist_ok=True)
(DIR_DRIVE / 'logs').mkdir(parents=True, exist_ok=True)
print('Checkpoints y logs en:', DIR_DRIVE)

## 3. Clonar el repositorio

In [ ]:
import os
from pathlib import Path

REPO = 'https://github.com/FerAHMz/space-invaders-dpl.git'
DIR_REPO = Path('/content/space-invaders-dpl')

if DIR_REPO.exists():
    !cd {DIR_REPO} && git pull --ff-only
else:
    !git clone {REPO} {DIR_REPO}

os.chdir(DIR_REPO)
!git log --oneline -3

## 4. Instalar dependencias

Se instalan solo las del entorno de Atari. **PyTorch y numpy se dejan como vienen
en Colab**: reinstalarlos con las versiones fijadas en `requirements.txt` rompe la
compilación de CUDA que Colab trae preconfigurada.

In [ ]:
!pip install -q "gymnasium[atari,other]==1.3.0" "ale-py==0.12.1" imageio-ffmpeg

import gymnasium as gym, ale_py
gym.register_envs(ale_py)
print('gymnasium', gym.__version__, '| ale-py', ale_py.__version__)
# Las ROMs vienen incluidas en ale-py desde la version 0.10.
env = gym.make('ALE/SpaceInvaders-v5')
print('entorno ok:', env.observation_space, env.action_space)
env.close()

## 5. Apuntar `modelos/` y `logs/` a Drive

El código escribe siempre en `modelos/` y `logs/` dentro del repo. En vez de
tocar el código, se reemplazan esas carpetas por enlaces simbólicos a Drive: así
el mismo script funciona igual en local y en Colab, y los checkpoints sobreviven
a las desconexiones.

In [ ]:
import shutil
from pathlib import Path

for nombre in ['modelos', 'logs']:
    destino = Path('/content/space-invaders-dpl') / nombre
    if destino.is_symlink():
        destino.unlink()
    elif destino.exists():
        shutil.rmtree(destino)
    destino.symlink_to(DIR_DRIVE / nombre)

!ls -la /content/space-invaders-dpl | grep -E "modelos|logs"
!ls -la {DIR_DRIVE}/modelos

## 6. Prueba rápida antes de la corrida larga

Verifica que el agente, el entorno y la GPU funcionan juntos, para no descubrir un error a las dos horas de entrenamiento.

In [ ]:
import sys
sys.path.insert(0, '/content/space-invaders-dpl/src')

from agente import AgenteRainbow, elegir_dispositivo
from wrappers import crear_entorno_entrenamiento

dispositivo = elegir_dispositivo()
print('dispositivo elegido:', dispositivo)

env = crear_entorno_entrenamiento(semilla=42)
obs, _ = env.reset(seed=42)
agente = AgenteRainbow(env.action_space.n, dispositivo, capacidad_replay=2000)

parametros = sum(p.numel() for p in agente.red.parameters())
print('observacion:', obs.shape, obs.dtype)
print('accion de ejemplo:', agente.actuar(obs))
print(f'parametros de la red: {parametros:,}')
env.close()

## 7. Entrenar

`--hasta` es un objetivo **absoluto**: si la sesión se corta, volver a ejecutar
esta celda continúa hacia los mismos 12 M de pasos en vez de agregar 12 M más.

12 M pasos de agente = 48 M frames. En una T4 son del orden de 10–14 h; en una
A100, 4–6 h. Se puede bajar el objetivo si tienes menos tiempo — el checkpoint
`_mejor.pt` siempre guarda la mejor evaluación vista hasta el momento, así que
cortar antes no deja el trabajo inservible.

In [ ]:
!cd /content/space-invaders-dpl && python -u src/02_entrenar_dqn.py \
    --etiqueta rainbow_colab \
    --hasta 12000000 \
    --eval-cada 250000 \
    --eval-episodios 5 \
    --guardar-cada 100000 \
    --reanudar

## 8. Evaluar con la configuración de la competencia

5 episodios, política greedy, sin clipping de recompensa. Se evalúa el checkpoint de mejor evaluación, no el último.

In [ ]:
!cd /content/space-invaders-dpl && python -u src/03_evaluar_agente.py \
    --modelo modelos/rainbow_colab_mejor.pt \
    --salida entregables/evaluacion_colab.json

## 9. Grabar el video entregable

In [ ]:
!cd /content/space-invaders-dpl && python -u src/04_generar_video_agente.py \
    --modelo modelos/rainbow_colab_mejor.pt \
    --episodios 5 \
    --prefijo agente-entrenado-space-invaders

In [ ]:
import json
from pathlib import Path
from IPython.display import HTML
from base64 import b64encode

meta = json.loads(Path('/content/space-invaders-dpl/entregables/video_agente.json').read_text())
mejor = meta['mejor']
print(f"Mejor partida: {mejor['recompensa_total']:.0f} puntos ({mejor['video']})")

ruta = Path('/content/space-invaders-dpl/entregables/videos') / mejor['video']
datos = b64encode(ruta.read_bytes()).decode()
HTML(f'<video width=420 controls><source src="data:video/mp4;base64,{datos}" type="video/mp4"></video>')

## 10. Curvas de entrenamiento

In [ ]:
!cd /content/space-invaders-dpl && python -u src/05_graficar_curvas.py --etiquetas rainbow_colab

from IPython.display import Image
Image('/content/space-invaders-dpl/entregables/figuras/curvas_rainbow_colab.png')

## 11. Guardar los entregables en Drive

Los checkpoints ya están en Drive por los enlaces del paso 5. Aquí se copian
además el video, la evaluación y las curvas, para bajarlos a la Mac y
versionarlos en el repo.

In [ ]:
import shutil
from pathlib import Path

origen = Path('/content/space-invaders-dpl/entregables')
destino = DIR_DRIVE / 'entregables'
if destino.exists():
    shutil.rmtree(destino)
shutil.copytree(origen, destino)

print('Entregables copiados a', destino)
for p in sorted(destino.rglob('*')):
    if p.is_file():
        print(f'  {p.relative_to(destino)}  ({p.stat().st_size/1e6:.1f} MB)')

---

### Bajar el modelo a la Mac

Desde la carpeta del repo en local:

```bash
# el checkpoint queda en Drive: MyDrive/space-invaders-dpl/modelos/rainbow_colab_mejor.pt
cp ~/Google\ Drive/MyDrive/space-invaders-dpl/modelos/rainbow_colab_mejor.pt \
   modelos/rainbow_space_invaders.pt

python src/03_evaluar_agente.py --modelo modelos/rainbow_space_invaders.pt
```

La evaluación local debe dar los mismos puntajes que en Colab: el
preprocesamiento viaja dentro del checkpoint, así que no puede desalinearse
entre las dos máquinas.